# 🔬 Notebook 3 — Deploy Strategies & SLO-gated Auto-Rollback

You have an immutable artifact (Notebook 2). Now you have to get it onto running servers **without breaking users**. This notebook walks through the three classic strategies — **rolling**, **blue/green**, **canary** — from worst to best, and then shows a tiny **automatic rollback** driven by error-rate metrics.

## 🛠️ Setup

```bash
cd 06-system-designs/code-deployment
uv sync
```

Pick the `.venv` kernel. Reload the window if needed.

## 1. The bad baseline — "big-bang" deploy

Stop *all* old instances, then start *all* new ones. Simple. Also: 100% downtime + no easy rollback.

In [ ]:
# 🔴 BAD: big-bang. Zero overlap ⇒ full outage window.
import time

class BigBang:
    def __init__(self, n=4):
        self.fleet = [{"id": i, "version": "v1", "healthy": True} for i in range(n)]

    def deploy(self, new_version: str):
        print("stopping ALL old instances…")
        for s in self.fleet:
            s["healthy"] = False
        time.sleep(0.1)
        print("⚠️  user traffic is 100% erroring right now")
        print("starting ALL new instances…")
        for s in self.fleet:
            s["version"] = new_version
            s["healthy"] = True
        print("done:", self.fleet)

BigBang().deploy("v2")

**Don't do this in production.** Use one of the three strategies below.

## 2. Rolling deploy — replace a few at a time

- Replace **N** instances at a time, wait for them to report healthy, then move on.
- At any moment, most of the fleet still serves traffic.
- Rollback = run a rolling deploy *back* to the previous version — which takes as long as
  the forward deploy did, and leaves you serving a **mixed fleet** the whole time.

We'll deploy a **broken** version so the abort path actually executes. This is the case
worth watching: a rolling deploy that fails halfway leaves the fleet split across two
versions, and getting out of that state is not free.

In [ ]:
# 🟡 ROLLING: batch-by-batch with health checks
from dataclasses import dataclass

STARTUP_S = 0.02          # pretend an instance takes this long to boot + pass /healthz

@dataclass
class Instance:
    id: int
    version: str
    healthy: bool = True
    draining: bool = False        # pulled from the LB, not yet replaced

def start(inst: Instance, version: str, broken_versions=()) -> bool:
    """Boot an instance on `version` and probe it. Returns True if it comes up healthy."""
    time.sleep(STARTUP_S)                       # boot + warm-up + /healthz
    inst.version = version
    inst.draining = False
    inst.healthy = version not in broken_versions
    return inst.healthy

def label(i: Instance) -> str:
    if i.healthy:   return i.version                    # serving traffic
    if i.draining:  return f"{i.version}(drained)"      # out of the LB, waiting its turn
    return f"{i.version}(sick)"                         # started up and failed its probe

def fleet_state(fleet):
    from collections import Counter
    return dict(Counter(label(i) for i in fleet))

def rolling(fleet, new_version, batch=2, broken_versions=()):
    """Returns (ok, seconds, capacity_low_water_mark)."""
    t0 = time.time()
    min_serving = len(fleet)
    for i in range(0, len(fleet), batch):
        window = fleet[i:i + batch]
        for w in window:
            w.healthy, w.draining = False, True  # drained from the load balancer first
        min_serving = min(min_serving, sum(1 for f in fleet if f.healthy))
        for w in window:
            w.draining = False
            if not start(w, new_version, broken_versions):
                print(f"  ❌ instance {w.id} failed health check on {new_version} — halting rollout")
                print(f"     fleet is now MIXED: {fleet_state(fleet)}")
                return False, time.time() - t0, min_serving
        print(f"  ✅ batch {[w.id for w in window]} -> {new_version}   fleet: {fleet_state(fleet)}")
    return True, time.time() - t0, min_serving


print("— happy path —")
fleet = [Instance(i, "v1") for i in range(6)]
ok, secs, low = rolling(fleet, "v2", batch=2)
print(f"result={ok}  took {secs:.2f}s  lowest serving capacity during rollout: {low}/6\n")

print("— v3 is broken —")
fleet = [Instance(i, "v2") for i in range(6)]
ok, secs, low = rolling(fleet, "v3", batch=2, broken_versions={"v3"})
print(f"result={ok}  took {secs:.2f}s\n")

print("— now roll BACK to v2: a second full rolling deploy —")
ok, back_secs, low = rolling(fleet, "v2", batch=2)
print(f"rollback took another {back_secs:.2f}s; total time in a bad/mixed state: {secs + back_secs:.2f}s")

**What the run shows**

- The rollout stopped at the **first** bad batch — good, only 1 of 6 instances actually
  came up broken.
- But the fleet is now **mixed and short-handed**: one `v3(sick)`, one `v2(drained)` that
  was pulled from the load balancer and never got its replacement, and four healthy `v2`.
  You are serving on 4 of 6 instances with two versions of the code in play.
- Getting back to safety means a *second* rolling deploy. The total time spent in a bad
  state is roughly **2× the deploy duration**. At 6 fake instances that's a fraction of a
  second; on a 600-instance fleet with a 60s startup it is 20 minutes of degraded service.

**Pros:** simple, no extra capacity needed, works with any orchestrator.
**Cons:** during the rollout — and during the rollback — some users hit `v1` and some hit
`v2`, so your code and your database schema must tolerate *both versions running at once*.
Rollback is not instant, and its cost scales with fleet size.

## 3. Blue/green — two fleets, flip the switch

Keep the old fleet (`blue`) running. Stand up a *whole* new fleet (`green`) with the new version. Run smoke tests against green. Once happy, **atomically switch the load balancer** to point at green. Rollback is *instant* — just flip back.

In [ ]:
# 🟢 BLUE/GREEN
class BlueGreen:
    def __init__(self, size=6):
        self.size = size
        self.blue  = [Instance(i, "v2") for i in range(size)]
        self.green: list = []
        self.live  = "blue"

    def deploy(self, new_version, broken_versions=()):
        t0 = time.time()
        print(f"standing up a whole second fleet (green) @ {new_version}…")
        self.green = [Instance(100 + i, "none") for i in range(self.size)]
        # Every green instance boots and is smoke-tested BEFORE any user traffic.
        healthy = [start(g, new_version, broken_versions) for g in self.green]
        build_s = time.time() - t0
        if not all(healthy):
            print(f"  ❌ green is unhealthy ({healthy.count(False)}/{self.size} bad) — "
                  f"blue never stopped serving. Zero users affected.")
            self.green = []
            return False, build_s, 0.0
        t1 = time.time()
        self.live = "green"                      # one atomic LB pointer flip
        return True, build_s, time.time() - t1

    def rollback(self):
        t0 = time.time()
        self.live = "blue"                       # blue is still sitting there, warm
        return time.time() - t0


print("— broken version: blue/green catches it before anyone sees it —")
bg = BlueGreen()
ok, build_s, flip_s = bg.deploy("v3", broken_versions={"v3"})
print(f"  live is still: {bg.live}\n")

print("— good version —")
bg = BlueGreen()
ok, build_s, flip_s = bg.deploy("v4")
print(f"  built green in {build_s*1000:.0f} ms, then flipped the LB in {flip_s*1e6:.0f} µs")
print(f"  live: {bg.live}")
rb = bg.rollback()
print(f"  rollback: {rb*1e6:.0f} µs — blue was never torn down, so 'undo' is a pointer write")
print(f"  live: {bg.live}")
print()
print(f"⏱  rolling rollback: ~{back_secs*1000:.0f} ms   |   blue/green rollback: ~{rb*1e6:.0f} µs")
print("   That gap is what you are buying with 2x capacity.")

**The comparison that matters:** rolling rollback took *another full deploy*; blue/green
rollback was a pointer write. And when the version was broken, blue/green caught it during
smoke tests with **zero** user impact, where rolling had already poisoned a third of the fleet.

**Pros:** instant rollback, and a real gate — you smoke-test green before it ever sees a user.
**Cons, honestly:**

- **2× capacity for the duration of the deploy.** On a large fleet that is a real bill, and
  your cloud quota may simply say no.
- **The flip is atomic for *new* requests only.** In-flight requests, open WebSockets, and
  long-running jobs on blue don't teleport. You need connection draining and a grace period.
- **Stateful anything breaks the illusion.** Both fleets talk to the *same* database. If v4
  ran a migration, "just flip back to blue" means old code against a new schema. Blue/green
  gives you an instant *code* rollback and no *data* rollback at all — which is why
  forward-compatible migrations are non-negotiable.
- **Blue goes cold.** Keep it around too briefly and rollback isn't available; too long and
  you're paying double for hours.

## 4. Canary — let 1% of users test production for you

Start the new version alongside the old one and **shift a tiny fraction of traffic** to it. If metrics (error rate, latency, business KPI) stay healthy, ramp to 5%, 25%, 100%. Otherwise, roll back — only ~1% of users were exposed.

```
   100% v1  ──▶  99% v1 / 1% v2  ──▶  90/10  ──▶  50/50  ──▶  100% v2
                        ▲ bad metrics here ⇒ rollback, impact was 1%
```

In [ ]:
# 🔵 CANARY routing: decide which version a given request goes to
import random

def route(pct_v2: float) -> str:
    return "v2" if random.random() * 100 < pct_v2 else "v1"

random.seed(42)
counts = {"v1": 0, "v2": 0}
for _ in range(10_000):
    counts[route(1.0)] += 1         # 1% canary
print("1% canary split over 10k requests:", counts)

### 4.1 A realistic canary runner with SLO gate

Strategy: ramp through `[1, 5, 25, 50, 100] %`. After each step, measure the canary's error rate and compare to baseline. If it's worse than baseline by > `threshold`, **roll back automatically**.

In [ ]:
def simulate_requests(n: int, err_rate: float) -> list[bool]:
    """Return n booleans: True = error, False = success."""
    return [random.random() < err_rate for _ in range(n)]

def observed_error_rate(results: list[bool]) -> float:
    return (sum(results) / len(results)) if results else 0.0

def gate(baseline_err, canary_err, canary_n, baseline_n,
         threshold=0.5, min_requests=500):
    """
    Returns one of: 'rollback', 'promote', 'inconclusive'.

    Three outcomes, not two. A gate that can only say yes/no will happily 'promote'
    on 3 requests, which is how bad canaries get to 100%.
    """
    if canary_n < min_requests or baseline_n < min_requests:
        return "inconclusive"                       # not enough data to decide anything
    if canary_err > baseline_err * (1 + threshold):
        return "rollback"
    return "promote"

def canary_deploy(baseline_err, real_canary_err, steps=(1, 5, 25, 50, 100),
                  traffic_per_step=20_000, min_requests=500):
    print(f"true baseline err={baseline_err:.2%}   true canary err={real_canary_err:.2%}")
    exposed = 0                                     # requests that hit the bad build
    for pct in steps:
        canary_reqs   = int(traffic_per_step * pct / 100)
        baseline_reqs = traffic_per_step - canary_reqs
        c_err = observed_error_rate(simulate_requests(canary_reqs,   real_canary_err))
        b_err = observed_error_rate(simulate_requests(baseline_reqs, baseline_err))
        exposed += canary_reqs
        verdict = gate(b_err, c_err, canary_reqs, baseline_reqs, min_requests=min_requests)
        flag = {"rollback": "🚨", "promote": "  ", "inconclusive": "❔"}[verdict]
        print(f"  {flag} step {pct:>3}%  canary_n={canary_reqs:<6} "
              f"b_err={b_err:.2%}  c_err={c_err:.2%}  -> {verdict}")
        if verdict == "rollback":
            print(f"  rolled back at {pct}%. Blast radius: {exposed:,} requests "
                  f"({exposed/(traffic_per_step*len(steps)):.1%} of the window).")
            return "rolled_back"
    return "promoted"

# ── Case 1: canary is fine ────────────────────────────────────────────────
random.seed(1)
print("— healthy canary —")
print("result:", canary_deploy(baseline_err=0.01, real_canary_err=0.012), "\n")

# ── Case 2: canary is 5x worse ────────────────────────────────────────────
random.seed(2)
print("— broken canary —")
print("result:", canary_deploy(baseline_err=0.01, real_canary_err=0.05), "\n")

# ── Case 3: the trap. Same broken build, a lower-traffic service. ─────────
# No step ever gathers min_requests on BOTH sides, so every step is
# 'inconclusive' -- and the ramp walks a 5%-error build all the way to 100%.
random.seed(3)
print("— broken canary, but too little traffic per step —")
print("result:", canary_deploy(baseline_err=0.01, real_canary_err=0.05,
                               traffic_per_step=800))

### 4.2 Read case 3 again — that's the bug in most canary setups

A build with a **5× worse error rate** was promoted to 100%. The gate wasn't wrong; it was
never given enough data to fire. Every step came back `inconclusive`, and the ramp treated
"inconclusive" as "fine". If your gate has only two outcomes, this is what you have built —
you just can't see it, because the log line says `promote`.

Two more things the output shows:

- **In case 2, the 1% step is inconclusive too.** 200 requests is under the threshold, so
  the rollback actually lands at 5%. The comforting line "only 1% of users were exposed"
  is false for any service that doesn't do enormous traffic. The printed **blast radius**
  is the honest number.
- **At 100% there is no baseline left** (`b_err=0.00%`, `baseline_n=0`), so the last step is
  always inconclusive. Once you're fully ramped you have nothing to compare against —
  which is exactly when you most want a comparison. Real systems keep a permanent 1–5%
  holdback fleet on the old version for this reason.

That leaves you three levers, and you must pick at least one:

| Lever | Cost |
|---|---|
| Hold each step longer (**bake time**) | deploys take 30–60 min instead of 5 |
| Send more traffic to the canary | bigger blast radius when it *is* broken |
| Lower `min_requests` | more false rollbacks from noise; engineers stop trusting the gate |

There is no free option. Low-traffic services genuinely cannot canary on error rate —
they should use blue/green plus synthetic smoke tests instead.

### Other properties of a good gate

- **Relative, not absolute, threshold.** `canary_err > baseline_err × 1.5` adapts to a
  service that is normally noisy. An absolute `> 1%` rule fires all night on a service
  whose baseline is 0.9%.
- **Compare to the *current* baseline**, not yesterday's — if a dependency is having a bad
  day, both sides look bad and the relative comparison stays fair.
- **Watch several signals.** Error rate *and* p99 latency *and* one business KPI. A build
  that returns HTTP 200 with an empty cart is invisible to an error-rate gate.
- **Route stickily.** If a user bounces between v1 and v2 across requests you get
  inconsistent UI *and* a muddied experiment. Hash on user id, not per request.
- **Watch out for who your 1% are.** Routing by `random()` is uniform; routing by
  "internal employees first" biases the sample toward people with warm caches and
  forgiving expectations.

## 5. Pulling it together — which strategy when?

| Strategy | Downtime | Extra capacity | Rollback speed | Good for |
|---|---|---|---|---|
| Big-bang | 100% window | 0 | slow | 🚫 never (except toys) |
| Rolling  | 0 | 0 | slow (full rolldown) | stateless services, backward-compatible changes |
| Blue/green | 0 | 2× | **instant flip** | risky releases; easy smoke-test gate |
| Canary   | 0 | small | fast (small blast radius) | user-facing services with good metrics |

### Bonus concepts (not coded, but important)

- **Feature flags.** Deploy the code dark, then flip a flag to enable it for 1% → 100% of users. Decouples *deploy* from *release*.
- **Progressive delivery.** Canary + automated metric analysis (e.g. Argo Rollouts, Flagger).
- **Forward-compatible DB migrations.** Always ship a migration that the old *and* new code can run against, then clean up later. Without this, rollback is dangerous.
- **Deployment windows & freezes.** Friday-afternoon prod deploys are where legends retire early.

## 🔑 Key takeaways

1. **Never big-bang.** Pick rolling, blue/green, or canary based on risk and capacity.
2. **Canary + automated rollback** gives you the smallest blast radius per risky change.
3. Rollback safety comes from **immutable artifacts** (Notebook 2) + a way to **shift traffic** (this notebook).
4. Deploy ≠ release. **Feature flags** let you ship code dark and release it separately.

Congrats — you now have the mental model (and runnable snippets) of a real-world code-deployment system. 🚀